# 🍷 WineDB — Fine Wine & Vintages Starter Notebook

A quick tour of the **WineDB sample**: relational tables of fine wine producers, cuvees, vintage releases, **exact varietal blend percentages**, and organoleptic tasting descriptors — plus the same data as a relational SQLite database.

The full dataset (3,675 vintages · 106 wineries · lab chemistry · valuation indices) lives at [wine-database.pages.dev](https://wine-database.pages.dev/).

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Locate the dataset regardless of the exact mount folder name
BASE = "."
for root, dirs, files in os.walk("/kaggle/input"):
    if "wineries.csv" in files:
        BASE = root
        break
print("Data directory:", BASE)

wineries = pd.read_csv(f"{BASE}/wineries.csv")
wines = pd.read_csv(f"{BASE}/wines.csv")
vintages = pd.read_csv(f"{BASE}/vintages.csv")
blends = pd.read_csv(f"{BASE}/blends.csv")
tasting = pd.read_csv(f"{BASE}/tasting_profiles.csv")
appellations = pd.read_csv(f"{BASE}/appellations.csv")

for name, df in [("wineries", wineries), ("wines", wines), ("vintages", vintages),
                 ("blends", blends), ("tasting_profiles", tasting), ("appellations", appellations)]:
    print(f"{name:>17}: {df.shape[0]:>4} rows x {df.shape[1]} cols")

vintages.head()

## Priciest releases in the sample

`release_price_usd` is fully populated. `valuation_index_usd` (median secondary-market valuation) is stricter: it only exists when at least 3 independent price observations fall in a 12-month window — otherwise it is honestly `NULL`, so in this small sample it is mostly empty.

In [ ]:
top = (vintages.sort_values("release_price_usd", ascending=False)
       .head(10)[["winery_name", "wine_name", "vintage_year", "abv_percent", "release_price_usd", "valuation_index_usd"]])
top

## Exact varietal blend composition

Unlike flat wine datasets, WineDB stores the **exact percentage of each grape variety per vintage** (the SQLite version enforces that blends never sum past 100.001%).

In [ ]:
pick = blends[(blends.wine_name == "Opus One") & (blends.vintage_year == blends[blends.wine_name == "Opus One"].vintage_year.max())]
if pick.empty:
    first = blends.iloc[0]
    pick = blends[(blends.wine_name == first.wine_name) & (blends.vintage_year == first.vintage_year)]

label = f"{pick.iloc[0].winery_name} — {pick.iloc[0].wine_name} {pick.iloc[0].vintage_year}"
ax = pick.set_index("variety_name")["percent"].sort_values().plot.barh(figsize=(8, 3.5), color="#800020")
ax.set_title(f"Varietal blend: {label}")
ax.set_xlabel("% of blend")
ax.set_ylabel("")
plt.tight_layout(); plt.show()
print(f"Blend sum: {pick.percent.sum():.1f}%")

## Relational joins across the tables

The tables share clean keys (`vintage_id`, names), so multi-table questions are one `merge` away. (Prefer SQL? The same sample ships as a relational `winedb.sqlite` with foreign keys and blend-sum triggers on [the WineDB site](https://wine-database.pages.dev/).)

In [ ]:
# Which appellations produce the priciest wines in the sample?
joined = (vintages.merge(wines, left_on=["winery_name", "wine_name"], right_on=["winery_name", "name"])
          .groupby("appellation_name")
          .agg(avg_release_usd=("release_price_usd", "mean"), vintages=("vintage_id", "count"))
          .sort_values("avg_release_usd", ascending=False))
joined.head(8).style.format({"avg_release_usd": "${:,.0f}"})

## What do these wines taste like?

Controlled organoleptic descriptors are linked per vintage — ideal for flavor-based recommendation engines.

In [ ]:
ax = tasting.descriptor.value_counts().head(15).sort_values().plot.barh(figsize=(8, 5), color="#d4af37")
ax.set_title("Most common tasting descriptors in the sample")
ax.set_xlabel("vintages tagged")
plt.tight_layout(); plt.show()

## Going further

- **Full dataset** (3,675 vintages, SAQ/LCBO lab chemistry, Liv-ex style valuation medians, TTB COLA label registry, 11,960-row provenance ledger): [wine-database.pages.dev](https://wine-database.pages.dev/)
- **Data dictionary & sources**: [GitHub — WhiskyyDB/wine-database](https://github.com/WhiskyyDB/wine-database)
- License: sample is **CC-BY-4.0** — free for any use with attribution.